# Milestone 2: Data Quality dan Profiling

**Dataset:** Daily Rainfall Observations from 460 Stations and Flood Event Records across West Java, Indonesia (2020-2025)  
**Sumber:** BMKG + BNPB/BPBD (Mendeley Data DOI: 10.17632/d7j3s64vvr.3)  
**Tujuan:** Melakukan data cleaning, profiling dengan Polars + DuckDB, transformasi wide->long, penanganan missing values, outlier, deduplikasi, validasi, dan menyimpan hasil bersih ke `prosessed_data/`

## 1. Import Library & Setup Path

In [ ]:
from pathlib import Path
import duckdb
import polars as pl

ROOT = Path(r"C:\project_1\data analysis\tugas 1")
DATASET_DIR = ROOT / "dataset"
RAW_XLSX = DATASET_DIR / "01_Data-Ch-Jabar-2020-2025.xlsx"
META_XLSX = DATASET_DIR / "02_West_Java_Rainfall_Station_Metadata_Lengkap.xlsx"
OUT_DIR = ROOT / "prosessed_data"
OUT_PARQUET = OUT_DIR / "west_java_rainfall_long_clean.parquet"
OUT_CSV_SAMPLE = OUT_DIR / "west_java_rainfall_long_clean_sample_10000.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"Output dir: {OUT_DIR}")

## 2. Load Data Wide Format

In [ ]:
print("[1/9] Loading wide matrix from Excel...")
df_wide = pl.read_excel(RAW_XLSX, sheet_name="DATA")
print(f"Wide shape: {df_wide.shape[0]:,} rows x {df_wide.shape[1]:,} cols")
df_wide.head()

## 3. Identifikasi Kolom Tanggal & Stasiun

In [ ]:
date_col = df_wide.columns[0]
station_cols = [c for c in df_wide.columns if c != date_col]
print(f"Date col: {date_col}")
print(f"Jumlah stasiun: {len(station_cols)}")
print(f"Stasiun contoh: {station_cols[:5]}")

## 4. Transformasi Wide -> Long Format

Matriks 2,192 hari x 460 stasiun akan di-unpivot menjadi bentuk long: `(tanggal, station_id, rainfall_mm)`. Hasilnya akan menghasilkan ~1.008.320 baris sehingga memenuhi syarat **> 1.000.000 baris**.

In [ ]:
print("[2/9] Melting wide -> long format...")
df_long = df_wide.unpivot(
    index=[date_col],
    on=station_cols,
    variable_name="station_id",
    value_name="rainfall_mm"
).rename({date_col: "tanggal"})

df_long = df_long.with_columns([
    pl.col("tanggal").cast(pl.Date),
    pl.col("rainfall_mm").cast(pl.Float64)
])

print(f"Long shape (raw melt): {df_long.shape[0]:,} rows x {df_long.shape[1]:,} cols")
print(f"Expected: {len(df_wide):,} * {len(station_cols)} = {len(df_wide)*len(station_cols):,} rows")
df_long.head()

## 5. Join dengan Metadata Stasiun

In [ ]:
print("[3/9] Loading station metadata & joining...")
try:
    df_meta = pl.read_excel(META_XLSX, sheet_name="metadata")
    df_meta = df_meta.rename({c: c.strip().lower().replace(" ", "_").replace("/", "_") for c in df_meta.columns})
    if "station_id" not in df_meta.columns and "id_stasiun" in df_meta.columns:
        df_meta = df_meta.rename({"id_stasiun": "station_id"})
    elif "station_id" not in df_meta.columns and "kode_stasiun" in df_meta.columns:
        df_meta = df_meta.rename({"kode_stasiun": "station_id"})
    elif "station_id" not in df_meta.columns and "nama_stasiun" in df_meta.columns:
        # try to match by name if needed - but better use existing
        pass
    print(f"Metadata shape: {df_meta.shape}")
    df_long = df_long.join(df_meta, on="station_id", how="left")
    print(f"After join shape: {df_long.shape}")
    print("Metadata joined successfully")
except Exception as e:
    print(f"[WARN] Gagal join metadata: {e}")
    print("Lanjut tanpa metadata")

df_long.head()

## 6. Profiling dengan DuckDB SUMMARIZE

In [ ]:
print("[4/9] Profiling with DuckDB SUMMARIZE...")
con = duckdb.connect()
con.register("df_long", df_long.to_pandas())
try:
    summ = con.execute("SUMMARIZE SELECT * FROM df_long").df()
    display(summ[["column_name","column_type","min","max","approx_unique","avg","count","null_percentage"]].head(20))
except Exception as e:
    print(f"[WARN] SUMMARIZE error: {e}")
finally:
    con.close()

## 7. Deteksi Missing Values

In [ ]:
print("[5/9] Detecting missing values...")
jumlah_baris = df_long.height
null_summary = (
    df_long.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
display(null_summary)
print(f"\nTotal null: {df_long.null_count().sum_horizontal().item():,}")
print("[INFO] rainfall_mm NaN dipertahankan sesuai prinsip raw observational data (README).")

## 8. Outlier Detection & Winsorization (IQR)

In [ ]:
print("[6/9] Outlier handling with IQR + winsorization...")
q = df_long.filter(pl.col("rainfall_mm").is_not_null() & (pl.col("rainfall_mm") >= 0)) \
          .select([
              pl.col("rainfall_mm").quantile(0.25).alias("q1"),
              pl.col("rainfall_mm").quantile(0.75).alias("q3")
          ]).row(0)
q1, q3 = q[0], q[1]
iqr = q3 - q1
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr
print(f"Q1={q1:.2f} | Q3={q3:.2f} | IQR={iqr:.2f}")
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")

df_out = df_long.with_columns([
    pl.when(pl.col("rainfall_mm").is_between(lower, upper, closed="both"))
      .then(pl.col("rainfall_mm"))
      .when(pl.col("rainfall_mm") > upper).then(upper)
      .when(pl.col("rainfall_mm") < lower).then(lower)
      .otherwise(pl.col("rainfall_mm"))
      .alias("rainfall_mm_clipped")
])
print("[INFO] rainfall_mm_clipped dibuat (winsorized/clipped)")
df_out.head()

## 9. Deduplikasi & Validasi

In [ ]:
print("[7/9] Deduplication & validation...")
key = ["tanggal", "station_id"]
before_dedup = df_out.height
df_dedup = df_out.unique(subset=key, keep="first")
after_dedup = df_dedup.height
print(f"Rows before dedup by {key}: {before_dedup:,}")
print(f"Rows after dedup: {after_dedup:,} | Removed: {before_dedup-after_dedup:,}")

valid = df_dedup.select([
    pl.len().alias("total_rows"),
    pl.col("rainfall_mm").is_null().sum().alias("null_rainfall"),
    pl.col("rainfall_mm_clipped").is_null().sum().alias("null_rainfall_clipped"),
    ((pl.col("rainfall_mm") < 0) & pl.col("rainfall_mm").is_not_null()).sum().alias("neg_rainfall"),
    pl.col("tanggal").is_null().sum().alias("null_tanggal"),
    pl.col("station_id").is_null().sum().alias("null_station_id"),
])
display(valid)

## 10. Simpan Data Bersih ke prosessed_data/

In [ ]:
print("[8/9] Preparing data to save...")
# Prioritaskan kolom penting
priority_cols = ["tanggal", "station_id", "rainfall_mm", "rainfall_mm_clipped"]
other_cols = [c for c in df_dedup.columns if c not in priority_cols]
df_save = df_dedup.select(priority_cols + other_cols)

print(f"Final shape: {df_save.shape[0]:,} rows x {df_save.shape[1]:,} cols")
df_save.head()

In [ ]:
print("[9/9] Saving processed data...")
df_save.write_parquet(OUT_PARQUET)
print(f"SAVED: {OUT_PARQUET}")
df_save.head(10000).write_csv(OUT_CSV_SAMPLE)
print(f"SAMPLE: {OUT_CSV_SAMPLE}")
print(f"\nFile size: {OUT_PARQUET.stat().st_size / (1024**2):.2f} MB")
print("\n" + "="*60)
print("MILESTONE 2 SELESAI")
print("="*60)
print(f"Total rows: {df_save.shape[0]:,} (> 1,000,000: {'YES' if df_save.shape[0] > 1000000 else 'NO'})")
print("Next: Lanjut ke Milestone 3 (notebooks/03_eda_and_insights.ipynb)")

## Catatan Milestone 2

- **Dimensi data**: Wide (2192 x 461) -> Long (~1,008,320 baris). Memenuhi syarat > 1.000.000 baris.
- **Missing values**: `rainfall_mm` memiliki nilai NaN (missing observation) sesuai README. Dipertahankan apa adanya (raw observational integrity). Tidak di-imputasi sembarangan.
- **Outlier**: Deteksi menggunakan metode IQR. Penanganan dengan **winsorization/clipping** (`rainfall_mm_clipped`) ke batas [Q1-1.5*IQR, Q3+1.5*IQR].
- **Deduplikasi**: Deduplikasi berdasarkan kunci bisnis `tanggal + station_id` dengan `keep='first'`.
- **Validasi**: Dicek null, nilai negatif, null pada kunci utama.
- **Tools**: Polars untuk transformasi/cleaning, DuckDB untuk `SUMMARIZE`.
- **Output**: Tersimpan di `prosessed_data/west_java_rainfall_long_clean.parquet` siap digunakan untuk Milestone 3.